In [1]:
# Keep scientific stack on NumPy 1.x for binary compatibility with this notebook environment
%pip install -q "numpy<2" "pandas<2.3" "shap<0.47" "xgboost==2.0.3" lightgbm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
import pickle
import warnings
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

# Load data
train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')

FEATURES = [
    'GridPosition', 'QualiGapToPole_pct', 'IsWet',
    'SafetyCarDeployed', 'AvgFinish_Last3', 'DNF_Rate_Last5',
    'PodiumRate_Last5', 'Constructor_AvgPts_Last3', 'SeasonProgress'
]
TARGET = 'Podium'

X_train = train[FEATURES]
y_train = train[TARGET]
X_val   = val[FEATURES]
y_val   = val[TARGET]

# Load pre-trained model; if missing, reproduce modelling notebook setup and save best pipeline
model_path = Path('../outputs/best_model.pkl')
if model_path.exists():
    with open(model_path, 'rb') as f:
        best_pipeline = pickle.load(f)
    print(f'Loaded model from: {model_path}')
else:
    preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

    candidates = {
        'Logistic Regression': Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=42
            ))
        ]),
        'XGBoost': Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                scale_pos_weight=scale_pos_weight,
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric='auc',
                verbosity=0
            ))
        ]),
        'LightGBM': Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', LGBMClassifier(
                class_weight='balanced',
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                verbose=-1
            ))
        ])
    }

    best_auc = -1.0
    best_name = None
    best_pipeline = None
    for name, pipe in candidates.items():
        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_prob)
        if auc > best_auc:
            best_auc = auc
            best_name = name
            best_pipeline = pipe

    model_path.parent.mkdir(parents=True, exist_ok=True)
    with open(model_path, 'wb') as f:
        pickle.dump(best_pipeline, f)
    print(f'No saved model found. Rebuilt from modelling setup and saved: {model_path}')
    print(f'Selected model: {best_name} (AUC-ROC={best_auc:.4f})')

print('Data and model loaded')
print(f'NumPy version: {np.__version__}')

Note: you may need to restart the kernel to use updated packages.
Loaded model from: ..\outputs\best_model.pkl
Data and model loaded
NumPy version: 1.26.4


In [2]:
# SHAP needs the raw XGBoost model, not the pipeline wrapper
# So we manually run the preprocessor first, then pass to SHAP

xgb_model = best_pipeline.named_steps['classifier']
preprocessor = best_pipeline.named_steps['preprocessor']

# Transform validation data through the preprocessor
X_val_processed = preprocessor.transform(X_val)

# Convert to DataFrame so SHAP shows feature names
X_val_processed_df = pd.DataFrame(
    X_val_processed, 
    columns=FEATURES
)

print(f"Processed validation set shape: {X_val_processed_df.shape}")
print("Ready for SHAP analysis")

Processed validation set shape: (360, 9)
Ready for SHAP analysis


In [3]:
# TreeExplainer is the correct explainer for XGBoost
# Much faster than KernelExplainer and exact (not approximate)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val_processed_df)

print(f"SHAP values computed")
print(f"Shape: {shap_values.shape}")
print(f"One SHAP value per observation per feature")
print(f"Positive SHAP = pushes prediction toward Podium")
print(f"Negative SHAP = pushes prediction away from Podium")

SHAP values computed
Shape: (360, 9)
One SHAP value per observation per feature
Positive SHAP = pushes prediction toward Podium
Negative SHAP = pushes prediction away from Podium


Runtime context restored
